# Ternary-Bonsai-2-27B-Abliterated-v2 (PQ2_0 + MTP) — 128K Context & No-Think Mặc Định (Kaggle GPU T4 x2 + Ngrok)

Notebook tối ưu cho **Kaggle GPU T4 x2** phục vụ mô hình **[BoldingBuilds/Ternary-Bonsai-2-27B-Abliterated-v2-PQ2_0-MTP-GGUF](https://huggingface.co/BoldingBuilds/Ternary-Bonsai-2-27B-Abliterated-v2-PQ2_0-MTP-GGUF)** với **Context 128K (`131072`)**, **tắt Thinking mặc định** và **Ngrok tên miền cố định**.

### Hướng dẫn trước khi nhấn Run All trên Kaggle
1. **Bật GPU**: *Session options* -> *Accelerator* -> chọn **GPU T4 x2** (cần 2x T4 để chứa 128K context).
2. **Bật Internet**: Bật **Internet ON**.
3. **Thêm & Tick Secrets**: Vào *Add-ons -> Secrets*, thêm và **tick chọn** `NGROK_AUTHTOKEN` và `NGROK_DOMAIN`.


## 1. Cấu hình (128K Context + Tắt Thinking mặc định) & Kiểm tra GPU


In [20]:
import os
import re
import subprocess
import sys

# ==========================================
# CẤU HÌNH MÔ HÌNH & SERVER
# ==========================================
REPO_ID = "BoldingBuilds/Ternary-Bonsai-2-27B-Abliterated-v2-PQ2_0-MTP-GGUF"
USE_MTP = True               # True: bản MTP (7.66 GB, decode nhanh hơn ~40%)
USE_VISION = True           # True: tải thêm mmproj-Q8_0 (0.63 GB) để nhận ảnh
CONTEXT_SIZE = 131072        # 128K context
KV_CACHE_TYPE = "q8_0"       # Nén KV cache 8-bit để 128K vừa mượt trong 2x T4 (30GB VRAM)

# BẬT / TẮT THINKING MẶC ĐỊNH
DEFAULT_THINKING = False     # False = tắt thinking mặc định; đổi thành True để bật lại mặc định
REASONING_EFFORT = "medium"  # Chỉ áp dụng khi DEFAULT_THINKING = True ("medium" hoặc "high")

SERVER_PORT = 8081
PRISM_RELEASE_TAGS = ["prism-b10828-a153697", "prism-b10743-adfffbe"]

MANUAL_NGROK_AUTHTOKEN = ""
MANUAL_NGROK_DOMAIN = ""

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else os.getcwd()
os.makedirs(WORK_DIR, exist_ok=True)
os.environ["PATH"] = "/opt/bin:/usr/local/cuda/bin:" + os.environ.get("PATH", "")

try:
    smi_full = subprocess.check_output(["nvidia-smi"], text=True)
    m_cuda = re.search(r"CUDA Version:\s*([0-9]+\.[0-9]+)", smi_full)
    DRIVER_CUDA = m_cuda.group(1) if m_cuda else "12.4"

    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
        text=True,
    ).strip()
    print("GPU phát hiện trên Kaggle:")
    print(smi)
    CUDA_ARCH = smi.splitlines()[0].split(",")[-1].strip().replace(".", "")
    print(f"CUDA Driver: {DRIVER_CUDA} | Context: {CONTEXT_SIZE} | Thinking mặc định: {DEFAULT_THINKING}")
except Exception as e:
    raise RuntimeError(
        "Không tìm thấy GPU NVIDIA! Vào Session options -> Accelerator -> chọn GPU T4 x2 rồi chạy lại."
    ) from e

GPU phát hiện trên Kaggle:
Tesla T4, 15360 MiB, 7.5
Tesla T4, 15360 MiB, 7.5
CUDA Driver: 13.0 | Context: 131072 | Thinking mặc định: False


## 2. Cài đặt thư viện & Tải `PrismML-Eng/llama.cpp` CUDA (~15 giây)


In [21]:
!pip install -q hf_transfer pyngrok nvidia-cuda-runtime-cu12 nvidia-cublas-cu12

import glob
import json
import shutil
import site
import tarfile
import urllib.request

BIN_DIR = os.path.join(WORK_DIR, "prism-bin")
CUDA_TAG_FILE = os.path.join(BIN_DIR, ".selected_cuda")
LLAMA_SERVER_BIN = None


def parse_ver(v_str):
    return tuple(int(x) for x in v_str.split("."))


def get_cuda_lib_paths(bin_dir):
    paths = [bin_dir, "/usr/local/nvidia/lib64", "/usr/local/cuda/lib64"]
    for sp in site.getsitepackages():
        paths.extend(glob.glob(os.path.join(sp, "nvidia", "*", "lib")))
    try:
        import torch
        paths.append(os.path.join(os.path.dirname(torch.__file__), "lib"))
    except Exception:
        pass
    seen = []
    for p in paths:
        if os.path.exists(p) and p not in seen:
            seen.append(p)
    return ":".join(seen)


def test_binary(bin_path):
    env = os.environ.copy()
    env["LD_LIBRARY_PATH"] = get_cuda_lib_paths(os.path.dirname(bin_path)) + ":" + env.get("LD_LIBRARY_PATH", "")
    res = subprocess.run([bin_path, "--list-devices"], env=env, capture_output=True, text=True)
    return res.returncode == 0, (res.stdout + "\n" + res.stderr).strip()


driver_ver = parse_ver(DRIVER_CUDA)

if os.path.exists(CUDA_TAG_FILE):
    with open(CUDA_TAG_FILE, "r") as f:
        saved_ver = parse_ver(f.read().strip())
    if saved_ver <= driver_ver:
        found_existing = glob.glob(os.path.join(BIN_DIR, "**", "llama-server"), recursive=True)
        if found_existing:
            LLAMA_SERVER_BIN = found_existing[0]

if LLAMA_SERVER_BIN and test_binary(LLAMA_SERVER_BIN)[0]:
    print(f"Đã có sẵn llama-server tương thích tại: {LLAMA_SERVER_BIN}")
else:
    LLAMA_SERVER_BIN = None
    for tag in PRISM_RELEASE_TAGS:
        if LLAMA_SERVER_BIN:
            break
        print(f"Đang kiểm tra bản prebuilt CUDA <= {DRIVER_CUDA} ở tag {tag}...")
        api_url = f"https://api.github.com/repos/PrismML-Eng/llama.cpp/releases/tags/{tag}"
        try:
            req = urllib.request.Request(api_url, headers={"User-Agent": "kaggle-bonsai-setup"})
            with urllib.request.urlopen(req, timeout=20) as resp:
                rel_data = json.loads(resp.read().decode("utf-8"))

            candidates = []
            for a in rel_data.get("assets", []):
                m = re.search(r"bin-linux-cuda-([0-9]+\.[0-9]+)-x64\.tar\.gz$", a["name"])
                if m:
                    c_ver = parse_ver(m.group(1))
                    if c_ver <= driver_ver:
                        candidates.append((c_ver, m.group(1), a))

            candidates.sort(key=lambda x: x[0], reverse=True)
            for _, ver_str, asset in candidates:
                url = asset["browser_download_url"]
                tar_path = os.path.join(WORK_DIR, asset["name"])
                print(f"Đang tải {asset['name']}...")
                urllib.request.urlretrieve(url, tar_path)

                if os.path.exists(BIN_DIR):
                    shutil.rmtree(BIN_DIR)
                os.makedirs(BIN_DIR, exist_ok=True)
                with tarfile.open(tar_path, "r:gz") as tf:
                    tf.extractall(BIN_DIR)
                os.remove(tar_path)

                found = glob.glob(os.path.join(BIN_DIR, "**", "llama-server"), recursive=True)
                if found:
                    cand = found[0]
                    os.chmod(cand, 0o755)
                    ok, out = test_binary(cand)
                    if ok:
                        LLAMA_SERVER_BIN = cand
                        with open(CUDA_TAG_FILE, "w") as f:
                            f.write(ver_str)
                        print(f"Tải & kiểm tra GPU thành công!\n{out}")
                        break
                    else:
                        print(f"Bản {asset['name']} không tương thích ({out[:200]}), thử bản khác...")
        except Exception as e:
            print(f"Lỗi khi kiểm tra tag {tag}: {e}")

    if not LLAMA_SERVER_BIN:
        LLAMA_DIR = os.path.join(WORK_DIR, "prism-llama.cpp")
        if os.path.exists(LLAMA_DIR):
            shutil.rmtree(LLAMA_DIR)
        print("Không có bản prebuilt phù hợp, đang clone & biên dịch PrismML-Eng/llama.cpp...")
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/PrismML-Eng/llama.cpp.git", LLAMA_DIR],
            check=True,
        )
        cmake_cmd = [
            "cmake", "-B", os.path.join(LLAMA_DIR, "build"), "-S", LLAMA_DIR,
            "-DCMAKE_BUILD_TYPE=Release", "-DGGML_CUDA=ON",
            f"-DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH}", "-DLLAMA_CURL=OFF",
        ]
        stub_cuda = "/usr/local/cuda/lib64/stubs/libcuda.so"
        if os.path.exists(stub_cuda):
            cmake_cmd.append(f"-DCUDA_cuda_driver_LIBRARY={stub_cuda}")
        subprocess.run(cmake_cmd, check=True)
        nproc = str(os.cpu_count() or 4)
        subprocess.run(
            ["cmake", "--build", os.path.join(LLAMA_DIR, "build"), "--target", "llama-server", "-j", nproc],
            check=True,
        )
        LLAMA_SERVER_BIN = os.path.join(LLAMA_DIR, "build", "bin", "llama-server")

Đã có sẵn llama-server tương thích tại: /kaggle/working/prism-bin/llama-prism-b10828-a153697/llama-server


## 3. Lấy mô hình (tự nhận từ `/kaggle/input` nếu có, hoặc tải siêu tốc từ Hugging Face)


In [22]:
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import hf_hub_download

MODEL_FILENAME = (
    "Ternary-Bonsai-2-27B-Abliterated-v2-PQ2_0-MTP.gguf"
    if USE_MTP
    else "Ternary-Bonsai-2-27B-Abliterated-v2-PQ2_0.gguf"
)

MODEL_PATH = None
if os.path.exists("/kaggle/input"):
    matches = glob.glob(f"/kaggle/input/**/{MODEL_FILENAME}", recursive=True)
    if matches:
        MODEL_PATH = matches[0]
        print(f"Phát hiện model sẵn trong Kaggle Dataset: {MODEL_PATH}")

if not MODEL_PATH:
    MODEL_DIR = os.path.join(WORK_DIR, "models")
    os.makedirs(MODEL_DIR, exist_ok=True)
    print(f"Đang tải {MODEL_FILENAME} từ {REPO_ID}...")
    MODEL_PATH = hf_hub_download(
        repo_id=REPO_ID,
        filename=MODEL_FILENAME,
        local_dir=MODEL_DIR,
    )
    print(f"Đã tải xong model: {MODEL_PATH}")

MMPROJ_PATH = None
if USE_VISION:
    MMPROJ_FILENAME = "Ternary-Bonsai-2-27B-mmproj-Q8_0.gguf"
    mm_matches = glob.glob(f"/kaggle/input/**/{MMPROJ_FILENAME}", recursive=True) if os.path.exists("/kaggle/input") else []
    if mm_matches:
        MMPROJ_PATH = mm_matches[0]
        print(f"Phát hiện mmproj sẵn trong Kaggle Dataset: {MMPROJ_PATH}")
    else:
        MODEL_DIR = os.path.join(WORK_DIR, "models")
        os.makedirs(MODEL_DIR, exist_ok=True)
        MMPROJ_PATH = hf_hub_download(
            repo_id=REPO_ID,
            filename=MMPROJ_FILENAME,
            local_dir=MODEL_DIR,
        )
        print(f"Đã tải xong mmproj: {MMPROJ_PATH}")

Đang tải Ternary-Bonsai-2-27B-Abliterated-v2-PQ2_0-MTP.gguf từ BoldingBuilds/Ternary-Bonsai-2-27B-Abliterated-v2-PQ2_0-MTP-GGUF...
Đã tải xong model: /kaggle/working/models/Ternary-Bonsai-2-27B-Abliterated-v2-PQ2_0-MTP.gguf
Đã tải xong mmproj: /kaggle/working/models/Ternary-Bonsai-2-27B-mmproj-Q8_0.gguf


## 4. Khởi chạy `llama-server` (128K Context + Cấu hình Thinking)


In [23]:
import socket
import time
import requests

subprocess.run(["pkill", "-f", "llama-server"], check=False)
time.sleep(1)

def is_port_free(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) != 0

while not is_port_free(SERVER_PORT):
    print(f"Cổng {SERVER_PORT} đang bận, thử cổng {SERVER_PORT + 1}...")
    SERVER_PORT += 1

LOG_PATH = "/tmp/llama_server.log"

if DEFAULT_THINKING:
    chat_kwargs = json.dumps({"reasoning_effort": REASONING_EFFORT})
    thinking_flags = []
else:
    chat_kwargs = json.dumps({"enable_thinking": False})
    thinking_flags = ["--reasoning-budget", "0"]

cmd = [
    LLAMA_SERVER_BIN,
    "-m", MODEL_PATH,
    "-ngl", "99",
    "-fa", "on",
    "-c", str(CONTEXT_SIZE),
    "--cache-type-k", KV_CACHE_TYPE,
    "--cache-type-v", KV_CACHE_TYPE,
    "--jinja",
    "--chat-template-kwargs", chat_kwargs,
    "--temp", "1.0",
    "--top-k", "20",
    "--top-p", "0.95",
    "--host", "127.0.0.1",
    "--port", str(SERVER_PORT),
] + thinking_flags

if USE_MTP:
    cmd.extend(["--spec-type", "draft-mtp", "--spec-draft-n-max", "2"])

if MMPROJ_PATH:
    cmd.extend(["--mmproj", MMPROJ_PATH])

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = get_cuda_lib_paths(os.path.dirname(LLAMA_SERVER_BIN)) + ":" + env.get("LD_LIBRARY_PATH", "")

print("Lệnh khởi chạy:", " ".join(cmd))
log_file = open(LOG_PATH, "w")
server_proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT, env=env)

print("Đang nạp mô hình vào VRAM (vui lòng đợi 20-60 giây)...")
ready = False
for _ in range(120):
    if server_proc.poll() is not None:
        with open(LOG_PATH, "r") as f:
            print(f.read()[-3000:])
        raise RuntimeError(f"llama-server bị thoát sớm với mã lỗi {server_proc.returncode}")
    try:
        r = requests.get(f"http://127.0.0.1:{SERVER_PORT}/health", timeout=2)
        if r.status_code == 200:
            ready = True
            break
    except Exception:
        pass
    time.sleep(2)

if not ready:
    with open(LOG_PATH, "r") as f:
        print(f.read()[-3000:])
    raise TimeoutError("llama-server không phản hồi /health sau 240 giây.")

print(f"llama-server đã sẵn sàng tại http://127.0.0.1:{SERVER_PORT}")

Lệnh khởi chạy: /kaggle/working/prism-bin/llama-prism-b10828-a153697/llama-server -m /kaggle/working/models/Ternary-Bonsai-2-27B-Abliterated-v2-PQ2_0-MTP.gguf -ngl 99 -fa on -c 131072 --cache-type-k q8_0 --cache-type-v q8_0 --jinja --chat-template-kwargs {"enable_thinking": false} --temp 1.0 --top-k 20 --top-p 0.95 --host 127.0.0.1 --port 8081 --reasoning-budget 0 --spec-type draft-mtp --spec-draft-n-max 2 --mmproj /kaggle/working/models/Ternary-Bonsai-2-27B-mmproj-Q8_0.gguf
Đang nạp mô hình vào VRAM (vui lòng đợi 20-60 giây)...
llama-server đã sẵn sàng tại http://127.0.0.1:8081


## 5. Mở Ngrok Tên miền Cố định (`NGROK_DOMAIN`) & Test API


In [24]:
from pyngrok import ngrok, conf
from pyngrok.installer import install_ngrok

def get_kaggle_secret(name, fallback=""):
    val = os.environ.get(name, "").strip()
    if val:
        return val
    try:
        from kaggle_secrets import UserSecretsClient
        val = UserSecretsClient().get_secret(name).strip()
        if val:
            return val
    except Exception:
        pass
    return fallback.strip()

NGROK_AUTHTOKEN = get_kaggle_secret("NGROK_AUTHTOKEN", MANUAL_NGROK_AUTHTOKEN)
NGROK_DOMAIN = get_kaggle_secret("NGROK_DOMAIN", MANUAL_NGROK_DOMAIN)
NGROK_DOMAIN = re.sub(r"^https?://", "", NGROK_DOMAIN).strip("/")

if not NGROK_AUTHTOKEN:
    raise ValueError(
        "Thiếu NGROK_AUTHTOKEN! Vào Add-ons -> Secrets, thêm và TICK chọn NGROK_AUTHTOKEN."
    )

ngrok.kill()
subprocess.run(["pkill", "-f", "ngrok"], check=False)
time.sleep(1)

ngrok_path = conf.get_default().ngrok_path
if not os.path.exists(ngrok_path):
    install_ngrok(ngrok_path)

subprocess.run([ngrok_path, "config", "add-authtoken", NGROK_AUTHTOKEN], check=True)

ngrok_cmd = [ngrok_path, "http", str(SERVER_PORT), "--log=stdout"]
if NGROK_DOMAIN:
    ngrok_cmd.extend(["--domain", NGROK_DOMAIN])
else:
    print("Cảnh báo: Chưa cấu hình NGROK_DOMAIN, đang mở tên miền ngẫu nhiên...")

ngrok_log = open("/tmp/ngrok.log", "w")
ngrok_proc = subprocess.Popen(ngrok_cmd, stdout=ngrok_log, stderr=subprocess.STDOUT)

PUBLIC_URL = None
for _ in range(30):
    if ngrok_proc.poll() is not None:
        with open("/tmp/ngrok.log", "r") as f:
            print(f.read()[-2000:])
        raise RuntimeError(f"ngrok bị thoát sớm với mã lỗi {ngrok_proc.returncode}")
    try:
        t_resp = requests.get("http://127.0.0.1:4040/api/tunnels", timeout=2)
        tunnels = t_resp.json().get("tunnels", [])
        https_tunnels = [t for t in tunnels if t.get("public_url", "").startswith("https://")]
        if https_tunnels:
            PUBLIC_URL = https_tunnels[0]["public_url"].rstrip("/")
            break
    except Exception:
        pass
    time.sleep(1)

if not PUBLIC_URL:
    with open("/tmp/ngrok.log", "r") as f:
        print(f.read()[-2000:])
    raise TimeoutError("Không lấy được PUBLIC_URL từ ngrok sau 30 giây.")

print("=" * 65)
print(f"PUBLIC NGROK URL : {PUBLIC_URL}")
print(f"OPENAI BASE URL  : {PUBLIC_URL}/v1")
print(f"MODEL ID         : {MODEL_FILENAME}")
print(f"CONTEXT SIZE     : {CONTEXT_SIZE} (KV Cache: {KV_CACHE_TYPE})")
print(f"THINKING MẶC ĐỊNH: {'BẬT' if DEFAULT_THINKING else 'TẮT'}")
print("HEADER BẮT BUỘC  : ngrok-skip-browser-warning: true (nếu dùng client mobile/custom)")
print("=" * 65)

print("\nĐang test gọi /v1/chat/completions qua Ngrok...")
test_resp = requests.post(
    f"{PUBLIC_URL}/v1/chat/completions",
    headers={
        "Content-Type": "application/json",
        "ngrok-skip-browser-warning": "true",
    },
    json={
        "model": MODEL_FILENAME,
        "messages": [{"role": "user", "content": "Reply with exactly: OK_READY"}],
        "max_tokens": 128,
    },
    timeout=120,
)
test_resp.raise_for_status()
data = test_resp.json()
print("Phản hồi từ server:", data["choices"][0]["message"]["content"])

Authtoken already set in configuration file: /root/.config/ngrok/ngrok.yml
PUBLIC NGROK URL : https://fedora-muzzle-neatly.ngrok-free.dev
OPENAI BASE URL  : https://fedora-muzzle-neatly.ngrok-free.dev/v1
MODEL ID         : Ternary-Bonsai-2-27B-Abliterated-v2-PQ2_0-MTP.gguf
CONTEXT SIZE     : 131072 (KV Cache: q8_0)
THINKING MẶC ĐỊNH: TẮT
HEADER BẮT BUỘC  : ngrok-skip-browser-warning: true (nếu dùng client mobile/custom)

Đang test gọi /v1/chat/completions qua Ngrok...
Phản hồi từ server: OK_READY


## 6. Tắt Server & Ngrok sạch sẽ (Không tự chạy khi `Run All`)

Trước khi nhấn **Stop Session** hoặc chuyển tài khoản Kaggle khác, hãy bỏ dấu `#` và chạy `stop_all()` để giải phóng Ngrok (tránh lỗi `ERR_NGROK_334`).


In [25]:
# def stop_all():
#     try:
#         ngrok.kill()
#     except Exception:
#         pass
#     subprocess.run(["pkill", "-f", "ngrok"], check=False)
#     subprocess.run(["pkill", "-f", "llama-server"], check=False)
#     print("Đã tắt Ngrok và llama-server sạch sẽ.")

# Bỏ dấu # ở dòng dưới khi muốn tắt server (KHÔNG bỏ dấu # nếu dùng Run All):
# stop_all()